# E16 — Does the audio INPUT REPRESENTATION limit audio-only fault-severity classification?

**Question.** Audio-only classification of the number of broken propellers (5 classes) on the TII UAV Realistic Fault
Dataset, tested on **unseen recording days** (leave-one-day-out). Is the score limited by the **input
representation**? Nine representations are trained with **everything else held fixed**: model, optimizer, epochs,
windows, tokens, splits and seeds.

**Reference (authoritative).** `TII_Audio_Only_Analysis_Mamba.ipynb` (the revised audio-only notebook). Its code is
copied **verbatim** for:
- setup and real `mamba-ssm`; constants, seeding and hooks;
- the metadata audit and AudioBuffer parsing; windows and tokens;
- the DAY-OUT split and per-fold train-only z-scoring;
- `AudioMamba`, `train_run`, and the mission-cluster bootstrap.

**Only these changes:**
- **(a)** The STANDARD protocol and every dependency on the v24/v28 notebooks are dropped (day-out only).
- **(b)** `AudioMamba` `in_dim` = the representation's feature dimension. The Mamba blocks are unchanged
  (874,496 params).
- **(c)** New feature builders (and the cache they need) plus the loop over representations.
- **(d)** A new analysis section.

**Out of scope.** No IMU, no fusion, no tuning. **Order-normalised / harmonic features are out of scope:** the
dataset has no rotor RPM, so frequencies cannot be expressed as multiples of the rotor rate.

**Facts kept in mind.**
- Each buffer = 256 frames × 6 interleaved slots. Slot 6 is identically zero (95/95), so 5 channels are kept. Slot 1
  is the loudest.
- The per-channel sample rate is **unverified** (~16 kHz hypothesis). Every frequency axis is a **per-channel FFT bin
  (unverified Hz)**, and no computation converts bins to Hz.

**SMOKE flag** (next cell, default `False`). When `True`: representations R0 and R5 only, day-out fold 1, seed 42,
1 epoch. Results go to separate `*_smoke` files and the pooled analysis is skipped.

Expected runtime on a Kaggle GPU: ~2–3.5 h (9 representations × 15 runs). Every run is resumable.

In [ ]:
# ==============================================================================
# E16 RUN FLAGS (the only switches a user needs)
# ==============================================================================
SMOKE = False        # True: reps R0 + R5 only, day-out fold 1, seed 42, 1 epoch (separate *_smoke outputs, no analysis)
SAVE_CKPT = False    # True: also save each run's best checkpoint (not needed for any result below)
ZIP_PROBS = True     # include e16_probs/ (test probabilities, needed to recompute every number) in e16_outputs.zip

## 0. Setup (reference cells 0a–0d, verbatim except output paths, the v24/v28 constants and the hook names)

In [ ]:
# ==============================================================================
# 0a. ENVIRONMENT, PATHS, DATASET
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must be set before the first cuBLAS call (determinism)

import ast
import copy
import csv
import gc
import glob
import hashlib
import json
import math
import random
import re
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
if IS_KAGGLE:
    BASE, OUT = Path("/kaggle/working/tii"), Path("/kaggle/working")
elif IS_COLAB:
    BASE, OUT = Path("/content/tii"), Path("/content")
else:
    BASE, OUT = Path.cwd() / "tii", Path.cwd() / "e16_outputs"
BASE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
ENV = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local")

REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p: Path) -> bool:
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_attached = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _attached += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _attached:
    DATA_ROOT, DATA_SOURCE = _attached[0], "attached Kaggle dataset"
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) into {_repo} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT, DATA_SOURCE = _repo / "Dataset", "git clone --depth 1"
assert _is_dataset_root(DATA_ROOT), f"{DATA_ROOT} does not contain class folders 0..4"

FIG_DIR = OUT / "figures"
CACHE_DIR = OUT / "e16_cache"                # E16: new cache (new arrays)
PROBS_DIR = OUT / "e16_probs"
CKPT_DIR = OUT / "e16_checkpoints"
RESULTS_CSV = OUT / "e16_results.csv"
for _d in (FIG_DIR, CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV} | BASE {BASE} | outputs {OUT}")
print(f"Dataset: {DATA_ROOT} ({DATA_SOURCE})")
print(f"Figures -> {FIG_DIR} | cache -> {CACHE_DIR} | results -> {RESULTS_CSV} | probs -> {PROBS_DIR}")

In [ ]:
# ==============================================================================
# 0b. REAL mamba-ssm (no fallback). Installs only if it is not importable already.
# ==============================================================================
import importlib
import importlib.util

import torch


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Real mamba-ssm needs a GPU: enable a GPU accelerator (Kaggle: Settings -> "
                       "Accelerator -> GPU) and re-run. There is deliberately no CPU fallback in this notebook.")

if importlib.util.find_spec("mamba_ssm") is None:
    print(f"Installing mamba-ssm (torch {torch.__version__}, CUDA {torch.version.cuda}, python "
          f"{sys.version_info.major}.{sys.version_info.minor}) ...")
    _ok = (_pip(["packaging", "ninja"])
           and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output above). Check that internet is ON and a GPU "
                           "is attached, then restart the session and re-run. No fallback model is used.")

try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"mamba_ssm could not be imported ({exc!r}). Restart the kernel after installation and re-run; "
                       "no fallback model is used.") from exc
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} imported: {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | "
      f"torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 0c. CONSTANTS — every fixed value lives here
# ==============================================================================
SEEDS = [42, 123, 2024]

# --- AudioBuffer layout: 1536 = 256 frames x 6 interleaved slots; sample i -> frame i // 6, slot i % 6 ---
N_SLOTS = 6
DROP_SLOT = 5                                   # 0-based slot index 5 (the 6th slot): identically zero, dropped
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
N_CH = len(KEEP_SLOTS)                          # 5 audio channels
FRAME = 256                                     # frames per buffer = samples per channel per buffer
BUFFER_LEN = N_SLOTS * FRAME                    # 1536
N_RFFT = FRAME // 2 + 1                         # 129 per-channel rFFT bins (0..128)
KEEP_BINS = list(range(1, 17))                  # model input: per-channel bins 1..16 (fixed a priori)
N_FEAT = N_CH * len(KEEP_BINS)                  # 80

# --- windows (chunk counts only: no IMU, no time axis) ---
WINDOW_CHUNKS = 128
STRIDE_CHUNKS = 64
TA = 64                                         # tokens per window
TOKEN_POOL = WINDOW_CHUNKS // TA                # 2 consecutive chunks averaged per token
STEADY = (0.10, 0.90)                           # steady flight = 10%..90% of a mission's chunks (by position)

ASSUMED_CHANNEL_RATE_HZ = 16000                 # UNVERIFIED, used ONLY for secondary axis labels, never in any computation

# --- dataset facts (asserted in section 1) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5                             # exclude if audio_start_offset / mission_duration >= this
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}        # the single remaining 12-29 mission joins the 12-31 group
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"

# --- model + training: transcribed from version 28/minor-project-v28.ipynb (config cell, UnimodalMamba, train_unimodal),
#     the audio arms A1/A2. E16: used unchanged; E16 does not read the v28 notebook. ---
D_MODEL = 256
DROPOUT = 0.15
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496                 # 2 x 437,248 (real mamba_ssm, incl. the pre-norm LayerNorms)
EPOCHS = 20                                     # fixed; no early stopping in v28
BATCH_SIZE = 32
LEARNING_RATE = 1e-4                            # AdamW
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6                                   # CosineAnnealingLR(T_max=EPOCHS, eta_min=LR_MIN), stepped per epoch
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
# checkpoint rule (v28): after every epoch, keep the state with the highest validation macro-F1 (strictly greater)

# --- splits ---
INNER_VAL_FRACTION, INNER_VAL_SEED = 0.15, 2027                    # v27 _split_inner_val (day-out protocol index 0)
INNER_VAL_FALLBACK_SEED = 2026                                     # used ONLY if v27's notebook cannot be read
V27_INNER_VAL_SHA1 = "58e3c037c513ace2"         # sha1[:16] of v27's _split_inner_val source (ast segment)

# --- analysis (fixed before seeing results) ---
N_PERMUTATIONS = 1000
PERM_SEED = 2029
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
TOP_CELLS = 10
SIGN_CONSISTENT_MIN_DAYS = 4
MIN_PER_CLASS_FOR_DAY_SIGN = 2
SPEARMAN_THRESHOLD = 0.7
PCA_BINS = list(range(1, 33))
EVIDENCE_K = [1, 2, 4, 8, 16, 32]

CHECK_DETERMINISM = True                        # train seed 42 for 3 epochs twice and compare losses exactly
DETERMINISM_EPOCHS = 3

HZ_PER_BIN = ASSUMED_CHANNEL_RATE_HZ / FRAME    # 62.5 Hz/bin — secondary axis labels ONLY
print(f"{N_SLOTS} slots x {FRAME} frames = {BUFFER_LEN} samples per buffer | channels kept {[s + 1 for s in KEEP_SLOTS]} "
      f"(slot {DROP_SLOT + 1} dropped) | model input {N_CH} ch x bins {KEEP_BINS[0]}-{KEEP_BINS[-1]} = {N_FEAT} features")
print(f"Windows: {WINDOW_CHUNKS} chunks, stride {STRIDE_CHUNKS}, {TA} tokens of {TOKEN_POOL} chunks | seeds {SEEDS}")

In [ ]:
# ==============================================================================
# 0c-E16. E16 CONSTANTS (representations, run plan, decision rule) — all fixed BEFORE any result is seen
# ==============================================================================
REP_DIMS = {"R0": 80, "R1": 320, "R2": 640, "R3": 80, "R4": 80, "R5": 272, "R7": 160, "R6a": 16, "R6b": 64}
RUN_ORDER = ["R0", "R1", "R2", "R3", "R4", "R5", "R7", "R6a", "R6b"]      # partial results stay usable
CANDIDATES = ["R1", "R2", "R3", "R4", "R5", "R7"]                          # K = 6 comparisons vs R0
DIAGNOSTIC_REPS = ["R6a", "R6b"]                                           # never candidates for the pass rule
BAND_EDGES = [1, 2, 3, 4, 5, 6, 8, 10, 13, 17, 22, 29, 38, 50, 66, 86, 129]   # R3: 16 log-spaced integer-bin bands
SHAPE_EPS = 1e-6                                                           # R4
LOGRATIO_EPS = 1e-6                                                        # cache: logratio
CONTINUITY_MAX_RATIO = 1.5                                                 # hook 7 (R7 runs only if it passes)

# ---- pre-registered decision rule (see the markdown cell before the analysis) ----
MIN_DELTA = 0.02                     # pooled paired window macro-F1 delta; measured seed noise floor ~0.016
PAIRED_CI = 99.0                     # 99% mission-cluster bootstrap CI because K = 6
MIN_FOLDS_SAME_SIGN = 4              # of 5 folds (seed-averaged)
REF_R0_WINDOW_F1 = (0.8072, 0.754, 0.859)   # audio-only notebook, day-out pooled window macro-F1 [95% CI] (hook 8)
REF_TOLERANCE = 0.02
VAR_PERMUTATIONS, VAR_PERM_SEED = 200, 2030   # descriptive variance shares: day labels permuted within class

# ---- run plan ----
if SMOKE:
    REPS_TO_RUN, FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN = ["R0", "R5"], [1], [42], 1
    RESULTS_CSV = OUT / "e16_results_smoke.csv"
    PROBS_DIR = OUT / "e16_probs_smoke"
    PROBS_DIR.mkdir(parents=True, exist_ok=True)
    CHECK_DETERMINISM = False
else:
    REPS_TO_RUN, FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN = RUN_ORDER, list(range(len(DAYS))), SEEDS, EPOCHS
MEANS_DIR = OUT / "e16_mission_means"
MEANS_DIR.mkdir(parents=True, exist_ok=True)
assert len(BAND_EDGES) == 17 and N_RFFT == 129
print(f"SMOKE={SMOKE} | reps {REPS_TO_RUN} | folds {FOLDS_TO_RUN} | seeds {SEEDS_TO_RUN} | epochs {EPOCHS_TO_RUN} | "
      f"results -> {RESULTS_CSV.name}, probs -> {PROBS_DIR.name}/ | SAVE_CKPT={SAVE_CKPT}")

In [ ]:
# ==============================================================================
# 0d. SEEDING / DETERMINISM, FILE-ACCESS LOGGER, PASS/FAIL HOOKS, FIGURE HELPERS
# ==============================================================================
def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed: int) -> torch.Generator:
    """DataLoader shuffling uses its own generator, so the shuffle order depends only on the seed."""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id: int) -> None:
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
set_all_seeds(SEEDS[0])
print("Determinism settings: cudnn.deterministic=True, cudnn.benchmark=False, "
      "torch.use_deterministic_algorithms(True, warn_only=True) (warn_only because mamba-ssm's custom CUDA kernels are "
      f"outside PyTorch's deterministic registry), CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']}, "
      "per-run seeding of random/numpy/torch/CUDA, a seeded torch.Generator for DataLoader shuffling, and worker_init_fn.")

# ---- dataset-file access logger: EVERY dataset file read in this notebook goes through open_logged ----
ACCESS_KINDS = ("AudioBuffer:first_record", "AudioBuffer:full_parse", "SensorCombined:first_last_timestamp")
FILE_LOG = Counter()
FILE_LOG_PER_MISSION = {k: Counter() for k in ACCESS_KINDS}


def open_logged(path, kind, mode="r"):
    assert kind in ACCESS_KINDS, f"unknown access kind {kind}"
    FILE_LOG[kind] += 1
    FILE_LOG_PER_MISSION[kind][Path(path).parent.name] += 1
    return open(path, mode) if "b" in mode else open(path, mode, encoding="utf-8")


# ---- verification hooks ----
HOOKS = defaultdict(list)
HOOK_NAMES = {1: "mission/class/day counts + exclusion set", 2: "slot 6 identically zero", 3: "split leakage + train-only z-score",
              4: "per representation: tokens (64, D), finite", 5: "identical train/val/test sets for every representation (paired)",
              6: "real mamba_ssm + params + determinism", 7: "R7 chunk-boundary continuity",
              8: "R0 reproduces the audio-only notebook (soft)"}


def hook(n, ok, msg, stop=None):
    ok = bool(ok)
    HOOKS[n].append((ok, msg))
    print(f"[HOOK {n}] {'PASS' if ok else 'FAIL'} — {msg}")
    stop = (n <= 4) if stop is None else stop
    if not ok and stop:
        raise RuntimeError(f"HOOK {n} FAILED: {msg}. Stopping before anything is trained.")
    return ok


# ---- figures (palette: validated categorical order; sequential = one hue; diverging = two hues + neutral midpoint) ----
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CLASS_MARKERS = ["o", "s", "^", "D", "v"]
DAY_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
DAY_MARKERS = ["o", "s", "^", "D", "v"]
SEQ_CMAP, DIV_CMAP = "Blues", "RdBu_r"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "grid.linewidth": 0.6, "lines.linewidth": 1.5, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})
FREQ_LABEL = "per-channel FFT bin (1-128)"
HZ_LABEL = "approx. Hz if the per-channel rate were 16 kHz (unverified)"
FREQ_CAVEAT = ("Frequency axis = per-channel FFT bin of the de-interleaved channels. The Hz axis assumes a per-channel "
               "rate of 16 kHz, which is UNVERIFIED (the files carry no sample rate).")
FIGURES = []


def add_bin_axis(ax, axis="x"):
    """Secondary Hz axis for a per-channel-bin axis. Label only: HZ_PER_BIN is never used in a computation."""
    fwd, inv = (lambda b: b * HZ_PER_BIN), (lambda h: h / HZ_PER_BIN)
    if axis == "x":
        sec = ax.secondary_xaxis("top", functions=(fwd, inv))
        sec.set_xlabel(HZ_LABEL, fontsize=7)
    else:
        sec = ax.secondary_yaxis("right", functions=(fwd, inv))
        sec.set_ylabel(HZ_LABEL, fontsize=7)
    sec.tick_params(labelsize=7)
    ax._bin_caveat = True
    return sec


def save_fig(fig, name, caveat, freq_axes=()):
    """Every figure is saved as PNG with its caveat printed under it. Frequency figures must carry the Hz caveat."""
    assert caveat, f"{name}: every figure needs a caveat"
    freq_ok = all(getattr(ax, "_bin_caveat", False) for ax in freq_axes) and (not freq_axes or "UNVERIFIED" in caveat)
    fig.text(0.5, -0.01, caveat, ha="center", va="top", fontsize=8, color="#52514e", wrap=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    FIGURES.append({"name": name, "freq": bool(freq_axes), "freq_caveat_ok": freq_ok})
    print(f"  saved {FIG_DIR / (name + '.png')}" + ("" if freq_ok else "  <- MISSING frequency caveat"))


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

## 1. Inventory, parse cache, integrity
- **1a — metadata audit** (reference, verbatim). This is the only place `SensorCombined` is opened, and only its
  first and last `timestamp` are read. The 4 corrupt missions are excluded, leaving 95.
- **1b — E16 parse cache** (new directory `e16_cache/`, one `.npz` per mission, written atomically). Each chunk is a
  buffer of modal length 1536. After removing each channel's per-chunk mean, the cache stores:
  - `pow` (n, 5, 129) = |rFFT|² over 256 frames, rectangular window (the reference's array);
  - `logratio` (n, 4, 16) = log10((P_k + 1e-6) / (P_1 + 1e-6)) for channels k = 2..5 vs channel 1, bins 1..16;
  - `cphase` (n, 4, 16, 2) = cos and sin of angle(X_k · conj(X_1)), bins 1..16;
  - `pow512` (n//2, 5, 257) = |rFFT|² of chunks (2j, 2j+1) concatenated per channel, mean removed over the 512
    frames, rectangular window;
  - boundary statistics for the continuity hook: the first and last **raw** sample of each chunk and channel, the
    per-chunk median |adjacent-sample difference|, and the original record index of each kept chunk.
- **1c / 1d** — counts (hook 1) and six-slot verification (hook 2), reference verbatim.
- **1e — continuity (hook 7).** For each channel:
  ratio = median |x[first of chunk i+1] − x[last of chunk i]| ÷ median |adjacent-sample difference within a chunk|.
  - Only pairs of chunks that were consecutive records are used.
  - The denominator is the median of the per-chunk medians.
  - If any channel's ratio is ≥ 1.5, R7 (which joins two chunks into one FFT) is **skipped** and reported as skipped.

In [ ]:
# ==============================================================================
# 1a. METADATA AUDIT — recording_started_time vs. SensorCombined first/last timestamp; exclusion rule
# ==============================================================================
def mission_day(name: str) -> str:
    d = name[8:18].replace("_", "-")            # rosbag2_YYYY_MM_DD-... -> YYYY-MM-DD
    return DAY_MERGE.get(d, d)


def find_one(mdir: Path, pattern: str):
    hits = sorted(mdir.glob(pattern))
    return hits[0] if hits else None


ALL_DIRS, LABEL_OF_DIR = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    try:
        lab = int(class_dir.name)
    except ValueError:
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF_DIR[mdir] = lab
print(f"Mission folders found: {len(ALL_DIRS)} | per class {np.bincount(list(LABEL_OF_DIR.values()), minlength=N_CLASSES).tolist()}")

_TS_RE = re.compile(rb'"timestamp"\s*:\s*(\d+)')


def sensor_first_last_timestamp(path):
    """First and last `timestamp` (microseconds) of a SensorCombined file: the first line, then a backwards scan of the
    file tail. Regex on raw bytes: no IMU signal value is decoded."""
    with open_logged(path, "SensorCombined:first_last_timestamp", "rb") as f:
        first = None
        for line in f:
            m = _TS_RE.search(line)
            if m:
                first = int(m.group(1))
                break
        f.seek(0, 2)
        size = f.tell()
        block, last = min(size, 1 << 16), None
        while last is None:
            f.seek(size - block)
            lines = f.read(block).splitlines()
            for line in reversed(lines[1:] if block < size else lines):   # lines[0] may be a partial line
                m = _TS_RE.search(line)
                if m:
                    last = int(m.group(1))
                    break
            if last is None and block == size:
                break
            block = min(size, block * 4)
    return first, last


def audio_first_record(path):
    with open_logged(path, "AudioBuffer:first_record") as f:
        for line in f:
            if line.strip():
                return json.loads(line)
    return None


_meta = []
for mdir in ALL_DIRS:
    a_path, s_path = find_one(mdir, "*AudioBuffer*.jsonl"), find_one(mdir, "*SensorCombined*.jsonl")
    assert a_path is not None and s_path is not None, f"{mdir.name}: missing AudioBuffer or SensorCombined file"
    rec = audio_first_record(a_path)
    assert rec is not None and "data" in rec and "recording_started_time" in rec, f"{mdir.name}: unexpected AudioBuffer keys"
    t0, t1 = sensor_first_last_timestamp(s_path)
    rst = int(rec["recording_started_time"])
    dur = (t1 - t0) / 1e6
    off = (rst - t0) / 1e6
    _meta.append({"mission": mdir.name, "class": LABEL_OF_DIR[mdir], "day": mission_day(mdir.name),
                  "recording_started_time": rst, "imu_first_us": t0, "imu_last_us": t1, "audio_offset_s": off,
                  "mission_duration_s": dur, "offset_over_duration": off / dur})
META = pd.DataFrame(_meta).set_index("mission")

EXCLUDED = set(META.index[META.offset_over_duration >= EXCLUDE_RATIO])
print("\nExcluded (offset / duration >= 0.5):")
print(META.loc[sorted(EXCLUDED), ["class", "day", "audio_offset_s", "mission_duration_s", "offset_over_duration"]].to_string())
_kept_meta = META.drop(index=list(EXCLUDED))
print(f"\nMax offset/duration among KEPT missions: {_kept_meta.offset_over_duration.max():.3f} "
      f"({_kept_meta.offset_over_duration.idxmax()}) — below {EXCLUDE_RATIO}")
print(f"Audio start offset among kept missions: median {_kept_meta.audio_offset_s.median():.2f} s, "
      f"range {_kept_meta.audio_offset_s.min():.2f}–{_kept_meta.audio_offset_s.max():.2f} s")
EXCLUSION_OK = (EXCLUDED == EXPECTED_EXCLUDED) and bool((_kept_meta.offset_over_duration < EXCLUDE_RATIO).all())
print(f"Excluded set == the 4 expected missions and no other mission has ratio >= {EXCLUDE_RATIO}: {EXCLUSION_OK}")

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
axs[0].hist(_kept_meta.audio_offset_s, bins=30, color=CLASS_COLORS[0], edgecolor="white", label="kept (95)")
axs[0].hist(META.loc[sorted(EXCLUDED), "audio_offset_s"].clip(upper=_kept_meta.audio_offset_s.max() * 3), bins=10,
            color=CLASS_COLORS[1], edgecolor="white", label="excluded (4, clipped)")
axs[0].set_xlabel("audio start offset (s) = (recording_started_time − first IMU timestamp) / 1e6")
axs[0].set_ylabel("missions")
axs[0].legend()
axs[1].scatter(META.mission_duration_s, META.offset_over_duration.clip(-0.2, 2), s=14,
               c=[CLASS_COLORS[1] if n in EXCLUDED else CLASS_COLORS[0] for n in META.index])
axs[1].axhline(EXCLUDE_RATIO, color="#52514e", lw=1, ls="--")
axs[1].set_xlabel("mission duration (s, first->last IMU timestamp)")
axs[1].set_ylabel("offset / duration (clipped to [-0.2, 2])")
fig.suptitle("Audio start offset relative to the IMU log (metadata only)")
save_fig(fig, "1a_audio_offset_distribution",
         "Only timestamps are used: recording_started_time (AudioBuffer) and the first/last SensorCombined timestamp. "
         "Dashed line = exclusion threshold 0.5.")

KEPT = sorted((d for d in ALL_DIRS if d.name not in EXCLUDED), key=lambda d: d.name)
LABEL = {d: LABEL_OF_DIR[d] for d in KEPT}                      # keyed by Path (v27's inner-val rule expects this)
DAY = {d: mission_day(d.name) for d in KEPT}
NAME2DIR = {d.name: d for d in KEPT}
LABEL_N = {d.name: LABEL[d] for d in KEPT}
DAY_N = {d.name: DAY[d] for d in KEPT}
print("\nState: no IMU signal values are read anywhere in this notebook — SensorCombined files are opened only here, "
      "and only their first and last 'timestamp' fields are extracted.")

In [ ]:
# ==============================================================================
# 1b. PARSE EACH KEPT AudioBuffer FILE ONCE -> e16_cache/<mission>.npz (skipped if the cache exists)
#     Reference parse loop; E16 stores pow, logratio, cphase, pow512 and boundary statistics.
# ==============================================================================
def chunk_arrays(raw):
    """raw: (n, 1536) integer buffers of the modal length. Pure numpy: the E16 per-chunk arrays."""
    n = len(raw)
    frames = raw.reshape(n, FRAME, N_SLOTS)                     # sample i -> frame i // 6, slot i % 6
    chf = frames[:, :, KEEP_SLOTS].astype(np.float64)           # (n, 256, 5) raw samples, channels 1..5
    out = {"first": chf[:, 0, :].astype(np.float32), "last": chf[:, -1, :].astype(np.float32),
           "within_med_absdiff": np.median(np.abs(np.diff(chf, axis=1)), axis=1).astype(np.float32)}
    ch = chf - chf.mean(axis=1, keepdims=True)                  # per-chunk mean removed, per channel
    X = np.fft.rfft(ch, axis=1)                                 # (n, 129, 5), rectangular window
    out["pow"] = (np.abs(X) ** 2).transpose(0, 2, 1).astype(np.float32)                      # (n, 5, 129)
    out["rms"] = np.sqrt((ch ** 2).mean(axis=1)).astype(np.float32)                          # (n, 5)
    Xb = X[:, 1:17, :]                                                                         # bins 1..16
    Pb = np.abs(Xb) ** 2
    out["logratio"] = np.log10((Pb[:, :, 1:] + LOGRATIO_EPS) / (Pb[:, :, :1] + LOGRATIO_EPS)).transpose(0, 2, 1) \
        .astype(np.float32)                                                                    # (n, 4, 16)
    ang = np.angle(Xb[:, :, 1:] * np.conj(Xb[:, :, :1])).transpose(0, 2, 1)                   # (n, 4, 16)
    out["cphase"] = np.stack([np.cos(ang), np.sin(ang)], axis=-1).astype(np.float32)          # (n, 4, 16, 2)
    m = n // 2
    pair = chf[: 2 * m].reshape(m, 2 * FRAME, N_CH)             # chunks (2j, 2j+1) concatenated in time, per channel
    pair = pair - pair.mean(axis=1, keepdims=True)              # mean removed over the 512 frames
    out["pow512"] = (np.abs(np.fft.rfft(pair, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)   # (m, 5, 257)
    return out


def parse_and_cache(mdir: Path) -> str:
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        return "cached"
    path = find_one(mdir, "*AudioBuffer*.jsonl")
    chunks, lens, rsts, n_bad = [], Counter(), set(), 0
    with open_logged(path, "AudioBuffer:full_parse") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                n_bad += 1
                continue
            x = rec.get("data")
            if x is None:
                n_bad += 1
                continue
            lens[len(x)] += 1
            chunks.append(x)
            rsts.add(rec.get("recording_started_time"))
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal} != {BUFFER_LEN}"
    assert len(rsts) == 1, f"{mdir.name}: recording_started_time is not constant ({len(rsts)} values)"
    keep_idx = np.array([i for i, x in enumerate(chunks) if len(x) == modal], dtype=np.int64)   # original record index
    raw = np.asarray([chunks[i] for i in keep_idx], dtype=np.int64)          # (n, 1536)
    del chunks
    n = len(raw)
    frames = raw.reshape(n, FRAME, N_SLOTS)
    slot_vals = frames.reshape(-1, N_SLOTS).astype(np.float64)
    arr = chunk_arrays(raw)
    vmin, vmax = int(raw.min()), int(raw.max())
    finite = bool(all(np.isfinite(arr[k]).all() for k in ("pow", "logratio", "cphase", "pow512")))
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, **arr, keep_idx=keep_idx, slot_std=slot_vals.std(axis=0), slot_mean=slot_vals.mean(axis=0),
             slot_zero=np.all(slot_vals == 0, axis=0), slot_absmax=np.abs(slot_vals).max(axis=0), n_chunks=n,
             mid_buffer=raw[n // 2].astype(np.int32), vmin=vmin, vmax=vmax, n_at_min=int((raw == vmin).sum()),
             n_at_max=int((raw == vmax).sum()), n_dropped=sum(v for k, v in lens.items() if k != modal),
             n_bad_lines=n_bad, modal_len=modal, recording_started_time=int(next(iter(rsts))), finite=finite)
    os.replace(tmp, cache)                                      # atomic: an interrupted parse never leaves a bad cache
    return "parsed"


_t0 = time.time()
PARSE_STATUS = {}
for i, mdir in enumerate(KEPT, 1):
    PARSE_STATUS[mdir.name] = parse_and_cache(mdir)
    if i % 10 == 0 or i == len(KEPT):
        print(f"  {i}/{len(KEPT)} missions | parsed {sum(v == 'parsed' for v in PARSE_STATUS.values())}, cached "
              f"{sum(v == 'cached' for v in PARSE_STATUS.values())} | {time.time() - _t0:.0f} s")

# ---- ONE load pass over the cache: compact arrays only (feature arrays are read per representation, later) ----
_SMALL_KEYS = ["n_chunks", "slot_std", "slot_mean", "slot_zero", "mid_buffer", "vmin", "vmax", "n_at_min", "n_at_max",
               "n_dropped", "n_bad_lines", "modal_len", "recording_started_time", "finite", "keep_idx", "first", "last",
               "within_med_absdiff"]
MISS = {}
for mdir in KEPT:
    with np.load(CACHE_DIR / f"{mdir.name}.npz") as z:
        v = {k: z[k] for k in _SMALL_KEYS}
    MISS[mdir.name] = {"n_chunks": int(v["n_chunks"]), "slot_std": v["slot_std"], "slot_mean": v["slot_mean"],
                       "slot_zero": v["slot_zero"], "mid_buffer": v["mid_buffer"], "vmin": int(v["vmin"]),
                       "vmax": int(v["vmax"]), "n_at_min": int(v["n_at_min"]), "n_at_max": int(v["n_at_max"]),
                       "n_dropped": int(v["n_dropped"]), "n_bad_lines": int(v["n_bad_lines"]),
                       "modal_len": int(v["modal_len"]), "finite": bool(v["finite"]),
                       "rst_cache": int(v["recording_started_time"]), "keep_idx": v["keep_idx"], "first": v["first"],
                       "last": v["last"], "within_med_absdiff": v["within_med_absdiff"]}
_dropped = sum(v["n_dropped"] for v in MISS.values())
print(f"\nModal buffer length = {BUFFER_LEN} in every mission: {all(v['modal_len'] == BUFFER_LEN for v in MISS.values())} | "
      f"non-modal chunks dropped: {_dropped} (in {sum(v['n_dropped'] > 0 for v in MISS.values())} missions) | "
      f"undecodable lines: {sum(v['n_bad_lines'] for v in MISS.values())}")
assert all(v["modal_len"] == BUFFER_LEN for v in MISS.values())
assert all(v["rst_cache"] == META.loc[k, "recording_started_time"] for k, v in MISS.items()), "cache/metadata mismatch"
print(f"Cache: {len(MISS)} missions, {sum(v['n_chunks'] for v in MISS.values()):,} chunks | load pass {time.time() - _t0:.0f} s total")

In [ ]:
# ==============================================================================
# 1c. COUNTS — missions, classes, days (HOOK 1)
# ==============================================================================
_cls = np.bincount([LABEL[d] for d in KEPT], minlength=N_CLASSES).tolist()
_days = [sum(DAY[d] == day for d in KEPT) for day in DAYS]
DAY_CLASS = pd.crosstab(pd.Series([DAY[d] for d in KEPT], name="day"), pd.Series([LABEL[d] for d in KEPT], name="class"))
DAY_CLASS = DAY_CLASS.reindex(index=DAYS, columns=range(N_CLASSES), fill_value=0)
print(f"Missions: {len(KEPT)} | per class {_cls} | per day {dict(zip(DAYS, _days))}")
print(f"Merged into 2021-12-31: {[d.name for d in KEPT if d.name[8:18] == '2021_12_29']}")
print("\nDay x class (missions):")
print(DAY_CLASS.assign(total=DAY_CLASS.sum(axis=1)).to_string())
print(f"Every class on every day: {bool((DAY_CLASS.values > 0).all())}")
hook(1, len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and EXCLUSION_OK and _days == EXPECTED_DAY_COUNTS
     and bool((DAY_CLASS.values > 0).all()),
     f"{len(KEPT)} missions, classes {_cls}, excluded set == the 4 names: {EXCLUSION_OK}, days {_days} "
     f"(expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, True, {EXPECTED_DAY_COUNTS})")

In [ ]:
# ==============================================================================
# 1d. SIX-SLOT VERIFICATION across all 95 missions (HOOK 2)
# ==============================================================================
_std = np.stack([MISS[d.name]["slot_std"] for d in KEPT])        # (95, 6)
_mean = np.stack([MISS[d.name]["slot_mean"] for d in KEPT])
_zero = np.stack([MISS[d.name]["slot_zero"] for d in KEPT])
_slot_tab = pd.DataFrame({"slot": [f"slot {s + 1}" for s in range(N_SLOTS)], "mean std": _std.mean(0).round(2),
                          "median std": np.median(_std, 0).round(2), "mean DC (slot mean)": _mean.mean(0).round(2),
                          "missions identically zero": _zero.sum(0),
                          "missions where largest std": np.bincount(_std.argmax(1), minlength=N_SLOTS)}).set_index("slot")
print("Per-slot statistics over all samples of each mission, then across the 95 missions:")
print(_slot_tab.to_string())
n_zero6 = int(_zero[:, DROP_SLOT].sum())
n_slot1_max = int((_std.argmax(1) == 0).sum())
print(f"\nSlot 6 identically zero in {n_zero6}/{len(KEPT)} missions | slot 1 has the largest std in {n_slot1_max}/{len(KEPT)}")

_vmin = min(v["vmin"] for v in MISS.values())
_vmax = max(v["vmax"] for v in MISS.values())
_at_ext = [(k, v["n_at_min"], v["n_at_max"]) for k, v in MISS.items() if v["vmin"] == _vmin or v["vmax"] == _vmax]
print(f"\nClipping/saturation: observed global sample range [{_vmin}, {_vmax}] "
      f"(int16 full scale is [-32768, 32767]: {'reached' if _vmin <= -32768 or _vmax >= 32767 else 'NOT reached'})")
print(f"  missions touching the global extreme: {len(_at_ext)}; samples sitting exactly at their own min/max per mission "
      f"(median over missions): {np.median([v['n_at_min'] + v['n_at_max'] for v in MISS.values()]):.0f} "
      f"(a large count at the extreme would indicate clipping)")
print(f"NaN/Inf in any cached spectrum: {not all(v['finite'] for v in MISS.values())}")

_nch = pd.DataFrame({"mission": [d.name for d in KEPT], "class": [LABEL[d] for d in KEPT], "day": [DAY[d] for d in KEPT],
                     "chunks": [MISS[d.name]["n_chunks"] for d in KEPT]})
print("\nChunks per mission: " + _nch.chunks.describe()[["min", "25%", "50%", "75%", "max"]].round(0).to_string().replace("\n", " | "))
print("Chunks per class:")
print(_nch.groupby("class").chunks.agg(["count", "sum", "median", "min", "max"]).to_string())
_imp_ms = [(META.loc[d.name, "imu_last_us"] - META.loc[d.name, "recording_started_time"]) / 1e3 / MISS[d.name]["n_chunks"]
           for d in KEPT]
print(f"\nDESCRIPTIVE ONLY (not used anywhere): if audio ran from recording_started_time to the last IMU timestamp, one "
      f"buffer would last {np.percentile(_imp_ms, 5):.1f}–{np.percentile(_imp_ms, 95):.1f} ms (5th–95th pct, median "
      f"{np.median(_imp_ms):.1f} ms). 16 ms (16 kHz per channel) is consistent with this, but it is unverified. A 48 kHz "
      f"mono assumption would imply {BUFFER_LEN / 48:.0f} ms.")
hook(2, n_zero6 == len(KEPT) == EXPECTED_N_MISSIONS, f"slot 6 identically zero in {n_zero6}/{len(KEPT)} missions")

In [ ]:
# ==============================================================================
# 1e. CHUNK-BOUNDARY CONTINUITY (HOOK 7): decides whether R7 (two chunks joined into one 512-point FFT) runs
# ==============================================================================
def continuity_ratio(parts):
    """parts: iterable of (first (n,5), last (n,5), within_med_absdiff (n,5), keep_idx (n,)).
    Per channel: median |first[i+1] - last[i]| over pairs of consecutive records / median of per-chunk medians of
    |adjacent-sample difference|. Returns (ratio (5,), n_pairs)."""
    jumps, within = [], []
    for first, last, wmed, kidx in parts:
        adj = np.where(np.diff(kidx) == 1)[0]
        jumps.append(np.abs(first[adj + 1].astype(np.float64) - last[adj].astype(np.float64)))
        within.append(wmed.astype(np.float64))
    J, Wd = np.concatenate(jumps), np.concatenate(within)
    return np.median(J, axis=0) / np.maximum(np.median(Wd, axis=0), 1e-12), len(J)


CONT_RATIO, _n_pairs = continuity_ratio((MISS[n]["first"], MISS[n]["last"], MISS[n]["within_med_absdiff"],
                                         MISS[n]["keep_idx"]) for n in sorted(MISS))
print(f"Boundary jump / within-chunk step, per channel ({_n_pairs:,} consecutive chunk pairs): " +
      ", ".join(f"ch{c + 1} {r:.2f}" for c, r in enumerate(CONT_RATIO)))
R7_OK = hook(7, bool((CONT_RATIO < CONTINUITY_MAX_RATIO).all()),
             f"median boundary jump / median within-chunk step < {CONTINUITY_MAX_RATIO} on every channel "
             f"(max {CONT_RATIO.max():.2f})" + ("" if (CONT_RATIO < CONTINUITY_MAX_RATIO).all() else
                                               " — chunks are NOT continuous: R7 will be SKIPPED"), stop=False)

## 2. Windows (reference) and representations (E16)
- **Windows (reference, unchanged):** 128 chunks, stride 64, inside one mission. **64 tokens** per window, and every
  token covers a pair of consecutive chunks (global even alignment).
  - For every representation except R7, the token is the mean of its two chunks' features.
  - R7 uses `pow512[j]` directly. That is the FFT of exactly that pair of chunks, so there is no averaging.
- **Representations (fixed now; nothing is added after seeing results).** Frequency = **per-channel FFT bin
  (unverified Hz)**.

| rep | features per token | D | role |
|---|---|---|---|
| R0 CONTROL | pow[:, :, 1:17] ** 0.25, 5 ch × 16 bins (= the audio-only notebook's feature) | 80 | control |
| R1 WIDEBAND | bins 1..64, ** 0.25 | 320 | candidate |
| R2 FULLBAND | bins 1..128, ** 0.25 | 640 | candidate |
| R3 BANDS | 16 log-spaced bands per channel, edges [1,2,3,4,5,6,8,10,13,17,22,29,38,50,66,86,129], (band-mean power) ** 0.25 | 80 | candidate |
| R4 SHAPE-ONLY | (p / (Σp + 1e-6)) ** 0.25, p = bins 1..16 per chunk and channel (loudness removed) | 80 | candidate |
| R5 SPATIAL | R0 + logratio (64) + cphase cos/sin (128) | 272 | candidate |
| R7 FINE-RES | pow512[j, :, 1:33] ** 0.25, no pair averaging; only if hook 7 passes | 160 | candidate |
| R6a CH1-ONLY | channel 1, bins 1..16 | 16 | diagnostic |
| R6b CH2-5-ONLY | channels 2..5, bins 1..16 | 64 | diagnostic |

Each representation's `X_ALL` is built on the fly from the cache and freed after its 15 runs. Each representation
gets **hook 4** (tokens (64, D), finite) and **hook 5** (the same windows and the same train/val/test missions as
every other representation, so all comparisons are paired).

In [ ]:
# ==============================================================================
# 4. WINDOWS (window metadata; HOOK 4 window part) — tokens are built per representation below
# ==============================================================================
# E16: the window METADATA is built once here (exactly the reference loop); token arrays are built per representation.
NAMES = [d.name for d in KEPT]                                   # reference cell 2d
Y_M = np.array([LABEL_N[n] for n in NAMES])
DAY_M = np.array([DAY_N[n] for n in NAMES])
_wm = []
for n in NAMES:
    N = MISS[n]["n_chunks"]
    for i, s in enumerate(range(0, N - WINDOW_CHUNKS + 1, STRIDE_CHUNKS)):
        rel = (s + WINDOW_CHUNKS / 2) / N
        _wm.append({"window_id": f"{n}#{i:04d}", "mission": n, "cls": LABEL_N[n], "day": DAY_N[n], "start": s,
                    "end": s + WINDOW_CHUNKS, "mission_chunks": N, "rel_centre": rel,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady")})
WMETA = pd.DataFrame(_wm)
del _wm
WIN_ID2ROW = {w: i for i, w in enumerate(WMETA.window_id)}
MISSION_ROWS = {m: g.index.values for m, g in WMETA.groupby("mission", sort=False)}

_len_ok = bool(((WMETA.end - WMETA.start) == WINDOW_CHUNKS).all())
_inside = bool(((WMETA.start >= 0) & (WMETA.end <= WMETA.mission_chunks)).all())
_one_mission = WMETA.groupby("window_id").mission.nunique().max() == 1 and WMETA.window_id.is_unique
_even = bool((WMETA.start % TOKEN_POOL == 0).all())          # E16: global even token alignment
print(f"Windows: {len(WMETA)} | all window starts even (token = chunk pair 2j, 2j+1): {_even}")
print(f"Window = {WINDOW_CHUNKS} chunks ≈ {WINDOW_CHUNKS * FRAME / ASSUMED_CHANNEL_RATE_HZ:.2f} s ONLY IF a buffer is "
      f"{1000 * FRAME / ASSUMED_CHANNEL_RATE_HZ:.0f} ms (unverified)")
print("\nWindows per class x phase:")
print(pd.crosstab(WMETA.cls, WMETA.phase).reindex(columns=["start", "steady", "end"]).assign(total=lambda t: t.sum(1)).to_string())
print(f"Windows per mission: median {WMETA.groupby('mission').size().median():.0f}, "
      f"min {WMETA.groupby('mission').size().min()}, max {WMETA.groupby('mission').size().max()}")
hook(4, _len_ok and _inside and _one_mission and _even,
     f"{len(WMETA)} windows: every window has {WINDOW_CHUNKS} chunks ({_len_ok}), lies inside one mission ({_inside and _one_mission}), "
     f"even start ({_even}); tokens are checked per representation")

In [ ]:
# ==============================================================================
# 4b. E16 FEATURE BUILDERS (pure numpy): per-chunk features -> (64, D) tokens per 128-chunk window
# ==============================================================================
REQ_KEYS = {"R0": ["pow"], "R1": ["pow"], "R2": ["pow"], "R3": ["pow"], "R4": ["pow"], "R6a": ["pow"], "R6b": ["pow"],
            "R5": ["pow", "logratio", "cphase"], "R7": ["pow512"]}


def chunk_features(rep, z):
    """z: dict of cached arrays for ONE mission. Returns (n_chunks, D) float32 per-chunk features (all reps but R7)."""
    pw = z["pow"]                                                             # (n, 5, 129)
    n = len(pw)
    if rep == "R0":
        f = pw[:, :, 1:17] ** 0.25
    elif rep == "R1":
        f = pw[:, :, 1:65] ** 0.25
    elif rep == "R2":
        f = pw[:, :, 1:129] ** 0.25
    elif rep == "R3":
        f = np.stack([pw[:, :, a:b].mean(axis=-1) for a, b in zip(BAND_EDGES[:-1], BAND_EDGES[1:])], axis=-1) ** 0.25
    elif rep == "R4":
        p = pw[:, :, 1:17]
        f = (p / (p.sum(axis=-1, keepdims=True) + SHAPE_EPS)) ** 0.25
    elif rep == "R5":
        f = np.concatenate([(pw[:, :, 1:17] ** 0.25).reshape(n, -1), z["logratio"].reshape(n, -1),
                            z["cphase"].reshape(n, -1)], axis=1)
    elif rep == "R6a":
        f = pw[:, 0, 1:17] ** 0.25
    elif rep == "R6b":
        f = pw[:, 1:5, 1:17] ** 0.25
    else:
        raise ValueError(rep)
    return f.reshape(n, -1).astype(np.float32)                               # channel-major, as in the reference


def rep_windows(rep, z, starts):
    """(len(starts), 64, D) tokens. Token t of a window starting at chunk s covers chunks (s + 2t, s + 2t + 1)."""
    D = REP_DIMS[rep]
    if rep == "R7":
        g = (z["pow512"][:, :, 1:33] ** 0.25).reshape(len(z["pow512"]), -1).astype(np.float32)   # (n//2, 160)
        out = [g[s // TOKEN_POOL: s // TOKEN_POOL + TA] for s in starts]
    else:
        Fm = chunk_features(rep, z)
        out = [Fm[s:s + WINDOW_CHUNKS].reshape(TA, TOKEN_POOL, D).mean(axis=1) for s in starts]
    return np.stack(out).astype(np.float32) if out else np.zeros((0, TA, D), np.float32)


def build_rep(rep, names, starts_of, load):
    """X (n_windows, 64, D) in `names` / window order, mission-mean features (n_missions, D), window ids."""
    D = REP_DIMS[rep]
    blocks, means = [], np.zeros((len(names), D))
    for mi, n in enumerate(names):
        Xm = rep_windows(rep, load(n, REQ_KEYS[rep]), starts_of(n))
        blocks.append(Xm)
        means[mi] = Xm.reshape(-1, D).mean(axis=0)
    return np.concatenate(blocks), means


def load_cached(name, keys):
    with np.load(CACHE_DIR / f"{name}.npz") as z:
        return {k: z[k] for k in keys}


print("Feature dimensions: " + ", ".join(f"{r} D={d}" for r, d in REP_DIMS.items()))

## 3. Splits — DAY-OUT only (reference cell 5, with the STANDARD protocol removed)
- 5 folds. Fold k tests on day k (12-29 is merged into 12-31) and trains on the other four days. Within each fold,
  ~15% of the training missions per class are held out for validation.
- The validation hold-out uses v27's `_split_inner_val`, if that notebook is attached under `/kaggle/input` and its
  fingerprint matches. Otherwise the reference's fixed-seed-2026 fallback runs. **The rule that ran is printed.**
- Class weights and split hashes are computed once. **z-score statistics are computed per representation, from each
  fold's training windows only** (hook 3).

In [ ]:
# ==============================================================================
# 5. SPLITS — DAY-OUT only (v27's inner-validation rule if found, else the fixed-seed-2026 fallback) (HOOK 3)
# ==============================================================================
def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def find_ref_notebook(fname, version):
    cands = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        cands += [str(base / f"version {version}" / fname), str(base / f"Version {version}" / fname), str(base / fname)]
    cands += [f"/content/{fname}", f"/content/drive/MyDrive/{fname}"]
    return next((Path(c) for c in cands if Path(c).is_file()), None)


def extract_function(nb_path, fname):
    """Source of top-level function `fname` from a notebook's code cells (shell/magic lines are skipped)."""
    try:
        nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"  could not read {nb_path}: {exc!r}")
        return None
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = cell["source"] if isinstance(cell["source"], str) else "".join(cell["source"])
        src = "\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%")))
        try:
            tree = ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == fname:
                return ast.get_source_segment(src, node)
    return None


def src_sha(src):
    return hashlib.sha1(src.encode()).hexdigest()[:16]


# ---------------- DAY-OUT: v27's _split_inner_val ----------------
_p27 = find_ref_notebook("minor-project-v27.ipynb", 27)
_src27 = extract_function(_p27, "_split_inner_val") if _p27 else None
if _src27 is not None and src_sha(_src27) == V27_INNER_VAL_SHA1:
    _ns27 = {"np": np, "N_CLASSES": N_CLASSES, "LABEL": LABEL, "INNER_VAL_SEED": INNER_VAL_SEED,
             "INNER_VAL_FRACTION": INNER_VAL_FRACTION}
    exec(_src27, _ns27)
    inner_val = lambda pool, k: _ns27["_split_inner_val"](pool, 0, k)       # protocol index 0 = day_out in v27
    INNER_VAL_RULE = f"v27 _split_inner_val (read from {_p27}, fingerprint OK, seed [{INNER_VAL_SEED}, 0, fold])"
else:
    loud(("v27 notebook not found" if _p27 is None else f"v27 _split_inner_val unreadable or fingerprint mismatch at {_p27}")
         + f"\nFALLBACK inner-validation rule: same per-class ~15% (min 1) mission hold-out with fixed seed "
         f"{INNER_VAL_FALLBACK_SEED}.")

    def inner_val(pool, k):
        rng = np.random.default_rng([INNER_VAL_FALLBACK_SEED, k])
        train, val = [], []
        for c in range(N_CLASSES):
            ms = sorted((m for m in pool if LABEL[m] == c), key=lambda m: m.name)
            perm = rng.permutation(len(ms))
            n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
            val += [ms[i] for i in perm[:n_val]]
            train += [ms[i] for i in perm[n_val:]]
        return train, val

    INNER_VAL_RULE = f"FALLBACK per-class hold-out, seed {INNER_VAL_FALLBACK_SEED}"
print(f"Inner-validation rule: {INNER_VAL_RULE}")

FOLDS = {}
for k, day in enumerate(DAYS):
    test = [d for d in KEPT if DAY[d] == day]
    tr, va = inner_val([d for d in KEPT if DAY[d] != day], k)
    FOLDS[("day_out", k)] = {"train": sorted(d.name for d in tr), "val": sorted(d.name for d in va),
                             "test": sorted(d.name for d in test), "fold_day": day}

# ---- leakage asserts (HOOK 3, part 1) ----
_leak_ok, _day_ok = True, True
for key, f in FOLDS.items():
    tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
    _leak_ok &= not (tr & va) and not (tr & te) and not (va & te) and (tr | va | te) == set(NAMES)
    if key[0] == "day_out":
        _day_ok &= not ({DAY_N[n] for n in tr | va} & {DAY_N[n] for n in te}) and {DAY_N[n] for n in te} == {f["fold_day"]}
_partition = sorted(n for k in range(len(DAYS)) for n in FOLDS[("day_out", k)]["test"]) == sorted(NAMES)
hook(3, _leak_ok, "no mission is in more than one of train/val/test in any fold; each fold covers all 95 missions")
hook(3, _day_ok and _partition, "day-out: the test day never appears in train/val; the 5 test folds partition the 95 missions")

# ---- E16: indices + class weights once (representation-independent); z-scores per representation in prep_rep ----
FOLD_BASE = {}
for key, f in FOLDS.items():
    idx = {s: np.where(WMETA.mission.isin(f[s]).values)[0] for s in ("train", "val", "test")}
    counts = np.bincount(WMETA.cls.values[idx["train"]], minlength=N_CLASSES)
    cw = (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)      # v28: inverse class frequency
    FOLD_BASE[key] = {"idx": idx, "class_weights": cw, "train_counts": counts,
                      "train_missions_hash": hashlib.sha1("\n".join(sorted(set(f["train"]))).encode()).hexdigest()[:12],
                      "split_hash": hashlib.sha1(json.dumps({s: sorted(f[s]) for s in ("train", "val", "test")})
                                                 .encode()).hexdigest()[:12]}
SPLIT_HASH = hashlib.sha1("|".join(FOLD_BASE[k]["split_hash"] for k in sorted(FOLD_BASE)).encode()).hexdigest()[:16]
WINDOW_HASH = hashlib.sha1("\n".join(WMETA.window_id).encode()).hexdigest()[:16]

print("\n" + "=" * 130)
print(f"{'fold':<16}{'test':<20}{'missions tr/val/test':<22}{'test missions per class':<26}{'windows tr/val/test':<22}"
      f"{'class weights (from training windows)'}")
print("=" * 130)
for key, f in FOLDS.items():
    P = FOLD_BASE[key]
    per_class = np.bincount([LABEL_N[n] for n in f["test"]], minlength=N_CLASSES).tolist()
    print(f"{key[0] + ' ' + str(key[1]):<16}{f['fold_day']:<20}"
          f"{'/'.join(str(len(f[s])) for s in ('train', 'val', 'test')):<22}{str(per_class):<26}"
          f"{'/'.join(str(len(P['idx'][s])) for s in ('train', 'val', 'test')):<22}"
          f"{np.round(P['class_weights'], 3).tolist()}" + ("   <- SMALL fold (10 missions)" if f["fold_day"] == SMALL_FOLD_DAY else ""))
print(f"Split hash (all folds): {SPLIT_HASH} | window-id hash: {WINDOW_HASH} | inner-validation rule: {INNER_VAL_RULE}")


def prep_rep(X):
    """Per-fold z-score from TRAINING windows only (HOOK 3, part 2) for the current representation."""
    prep, ok = {}, True
    for key, base in FOLD_BASE.items():
        idx = base["idx"]
        used = set(WMETA.mission.values[idx["train"]])
        ok &= used == set(FOLDS[key]["train"]) and not (used & (set(FOLDS[key]["val"]) | set(FOLDS[key]["test"])))
        Xtr = X[idx["train"]].reshape(-1, X.shape[-1])
        mu = Xtr.mean(0, dtype=np.float64).astype(np.float32)
        sd = (Xtr.std(0, dtype=np.float64) + 1e-6).astype(np.float32)
        del Xtr
        prep[key] = {**base, "mu": mu, "sd": sd, "zhash": sha16(mu, sd)}
    return prep, ok

## 4. Model and training (reference cells 6a–6b, verbatim except `in_dim` and the representation name)
- Real `mamba_ssm.Mamba` only. v28's `UnimodalMamba` audio path: encoder `Linear(in_dim, 256) → LayerNorm → GELU →
  Linear(256, 256)` → **2 pre-norm residual Mamba blocks** (d_state 16, d_conv 2, expand 2; 874,496 parameters for
  every representation) → attention pooling → classifier.
- AdamW (lr 1e-4, weight decay 1e-4), cosine to 1e-6 over 20 epochs, grad clip 1.0, batch 32, **no early stopping**,
  checkpoint = the epoch with the best validation macro-F1, class-weighted CE.
- The determinism check (day-out fold 1, seed 42, 3 epochs, twice) runs on R0, before its runs.

In [ ]:
# ==============================================================================
# 6a. MODEL (v28 UnimodalMamba, audio path) + settings print (HOOK 6)
# ==============================================================================
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda")

# E16: no dependency on version 28's notebook; the reference's transcribed v28 settings are used unchanged.
V28_KEYS = ["D_MODEL", "DROPOUT", "MAMBA_NUM_BLOCKS", "MAMBA_D_STATE", "MAMBA_D_CONV", "MAMBA_EXPAND", "EXPECTED_MAMBA_PARAMS",
            "EPOCHS", "BATCH_SIZE", "LEARNING_RATE", "WEIGHT_DECAY", "LR_MIN", "GRAD_CLIP_NORM", "NUM_WORKERS"]
print("Training settings (reference = v28 A1): " + ", ".join(f"{k}={globals()[k]}" for k in V28_KEYS))


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM (v28)."""

    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(D_MODEL)
        self.mamba = MAMBA_CLASS(d_model=D_MODEL, d_state=MAMBA_D_STATE, d_conv=MAMBA_D_CONV, expand=MAMBA_EXPAND)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class AudioMamba(nn.Module):
    """v28 UnimodalMamba, audio path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention
    pooling -> classifier."""

    def __init__(self, in_dim=N_FEAT):
        super().__init__()
        self.audio_encoder = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, x):
        h = self.token_norm(self.audio_encoder(x) + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1))


def verify_model(model):
    real = all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks) and MAMBA_CLASS.__module__.startswith("mamba_ssm")
    blocks_ok = len(model.mamba_blocks) == MAMBA_NUM_BLOCKS and all(
        (b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND) for b in model.mamba_blocks)
    n_mamba = sum(p.numel() for p in model.mamba_blocks.parameters())
    n_params = sum(p.numel() for p in model.parameters())
    return real and blocks_ok and n_mamba == EXPECTED_MAMBA_PARAMS, n_params, n_mamba


set_all_seeds(SEEDS[0])
_probe = AudioMamba().to(device)
_ok, N_PARAMS, _n_mamba = verify_model(_probe)
assert _ok, "model is not the expected real-mamba_ssm architecture"
print(f"AudioMamba: {N_PARAMS:,} parameters (Mamba blocks {_n_mamba:,} == {EXPECTED_MAMBA_PARAMS:,}) | block class "
      f"{MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | modules {[n for n, _ in _probe.named_children()]}")
with torch.no_grad():
    assert _probe(torch.zeros(2, TA, N_FEAT, device=device)).shape == (2, N_CLASSES)
hook(6, _ok, f"every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); {N_PARAMS:,} parameters",
     stop=False)
del _probe
torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# 6b. TRAINING FUNCTION + optional determinism check (HOOK 6)
# ==============================================================================
def make_loader(X, y, shuffle, seed):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=worker_init_fn, generator=make_generator(seed) if shuffle else None)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = [F.softmax(model(x.to(device, non_blocking=True)), dim=-1).float().cpu().numpy() for x, _ in loader]
    return np.concatenate(out)


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(N_CLASSES))))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def mission_level(missions, y, P):
    """Mission prediction = argmax of the mean window probabilities."""
    df = pd.DataFrame(P, columns=range(N_CLASSES))
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g.y.first().values, g[list(range(N_CLASSES))].mean().values, g.y.first().index.values


def train_run(key, seed, epochs=EPOCHS, save=True, verbose=True):
    t0 = time.time()
    prep, fold = REP_PREP[key], FOLDS[key]
    idx = prep["idx"]
    Xs = {s: ((X_ALL[idx[s]] - prep["mu"]) / prep["sd"]).astype(np.float32) for s in ("train", "val", "test")}
    ys = {s: WMETA.cls.values[idx[s]].astype(np.int64) for s in ("train", "val", "test")}
    set_all_seeds(seed)
    model = AudioMamba(in_dim=X_ALL.shape[-1]).to(device)      # E16 (b)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()[:12]
    ok, n_params, _ = verify_model(model)
    assert ok, "not the expected real-mamba_ssm model"
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(prep["class_weights"], device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(Xs["train"], ys["train"], True, seed)
    va = make_loader(Xs["val"], ys["val"], False, seed)
    te = make_loader(Xs["test"], ys["test"], False, seed)
    tag = f"[{CUR_REP} | {key[0]} {key[1]} | seed {seed}]"
    if verbose:
        print(f"{tag} init weights {init_sha1} | {n_params:,} params | windows train {len(ys['train'])} / val "
              f"{len(ys['val'])} / test {len(ys['test'])} | z-score {prep['zhash']}")
    best_f1, best_state, best_epoch, history = -1.0, None, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        running, nb = 0.0, 0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running += loss.item()
            nb += 1
        scheduler.step()
        assert math.isfinite(running), f"{tag} non-finite training loss"
        val_f1 = compute_metrics(ys["val"], predict(model, va))["macro_f1"]
        history.append((running / nb, val_f1))
        improved = val_f1 > best_f1
        if improved:
            best_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        if verbose:
            print(f"{tag} epoch {epoch:02d}/{epochs} | train CE {running / nb:.4f} | val macro-F1 {val_f1:.4f}"
                  + ("  * best" if improved else ""))
    model.load_state_dict(best_state)
    P = predict(model, te)
    wm = compute_metrics(ys["test"], P)
    my, mP, _ = mission_level(WMETA.mission.values[idx["test"]], ys["test"], P)
    mm = compute_metrics(my, mP)
    name = f"{CUR_REP}_{key[1]}_{seed}"
    res = {"rep": CUR_REP, "D": int(X_ALL.shape[-1]), "inner_val_rule": INNER_VAL_RULE,
           "protocol": key[0], "fold": key[1], "fold_day": fold["fold_day"], "seed": seed,
           **{f"n_{s}_missions": len(fold[s]) for s in ("train", "val", "test")},
           **{f"n_{s}_windows": len(ys[s]) for s in ("train", "val", "test")},
           **wm, **{f"mission_{k}": v for k, v in mm.items()},
           "best_epoch": best_epoch, "best_val_macro_f1": round(best_f1, 6), "init_sha1": init_sha1,
           "zstats_sha1": prep["zhash"], "class_weights": json.dumps(np.round(prep["class_weights"], 4).tolist()),
           "n_params": n_params, "train_seconds": round(time.time() - t0, 1), "history": history}
    if save:
        m = WMETA.iloc[idx["test"]]
        np.savez(PROBS_DIR / f"{name}.npz", window_id=m.window_id.values.astype(str), mission=m.mission.values.astype(str),
                 cls=m.cls.values, day=m.day.values.astype(str), phase=m.phase.values.astype(str), start=m.start.values,
                 probs=P.astype(np.float32))
        if SAVE_CKPT:
            torch.save(best_state, CKPT_DIR / f"{name}.pt")
        res["probs_file"] = str(PROBS_DIR / f"{name}.npz")
        res["checkpoint"] = str(CKPT_DIR / f"{name}.pt") if SAVE_CKPT else ""
    del model, optimizer, scheduler, best_state, tr, va, te, Xs
    gc.collect()
    torch.cuda.empty_cache()
    return res


DETERMINISM = None


def run_determinism_check():
    """Reference determinism check (verbatim body), run on R0 because it needs a representation's X_ALL."""
    global DETERMINISM
    _key = ("day_out", 1)
    print(f"Determinism check: {_key}, seed 42, {DETERMINISM_EPOCHS} epochs, twice ...")
    _r1 = train_run(_key, 42, epochs=DETERMINISM_EPOCHS, save=False, verbose=False)
    _r2 = train_run(_key, 42, epochs=DETERMINISM_EPOCHS, save=False, verbose=False)
    DETERMINISM = _r1["history"] == _r2["history"] and _r1["init_sha1"] == _r2["init_sha1"]
    for e, (a, b) in enumerate(zip(_r1["history"], _r2["history"]), 1):
        print(f"  epoch {e}: run1 CE {a[0]:.8f} val-F1 {a[1]:.6f} | run2 CE {b[0]:.8f} val-F1 {b[1]:.6f}")
    hook(6, DETERMINISM, f"determinism: identical initial weights ({_r1['init_sha1']}) and bit-identical per-epoch losses "
                         f"over two {DETERMINISM_EPOCHS}-epoch runs" + ("" if DETERMINISM else
                         " — NOT identical: mamba-ssm's CUDA backward may use non-deterministic atomics; results are "
                         "then reproducible only up to that noise"), stop=False)

In [ ]:
# ==============================================================================
# 6c. RUNS — representation-major: build X_ALL from the cache, hooks 4/5/3, 5 folds x 3 seeds; CSV append per run;
#     resumable (key = rep/fold/seed); X_ALL freed after each representation. (Reference 6c loop, extended.)
# ==============================================================================
CSV_FIELDS = ["rep", "D", "protocol", "fold", "fold_day", "seed", "n_train_missions", "n_val_missions", "n_test_missions",
              "n_train_windows", "n_val_windows", "n_test_windows", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
              "mission_accuracy", "mission_macro_f1", "mission_balanced_accuracy", "mission_macro_auroc", "best_epoch",
              "best_val_macro_f1", "init_sha1", "zstats_sha1", "class_weights", "n_params", "train_seconds",
              "inner_val_rule", "probs_file", "checkpoint", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    return {(r.rep, int(r.fold), int(r.seed)): r for r in df.itertuples()
            if (PROBS_DIR / f"{r.rep}_{int(r.fold)}_{int(r.seed)}.npz").exists()}


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


SKIPPED_REPS = {}
_starts_of = lambda n: WMETA.start.values[MISSION_ROWS[n]]
_plan = [(r, k, s) for r in REPS_TO_RUN for s in SEEDS_TO_RUN for k in FOLDS_TO_RUN if not (r == "R7" and not R7_OK)]
_done = load_done()
_todo = [x for x in _plan if x not in _done]
print(f"{len(_plan)} runs planned | already in {RESULTS_CSV.name}: {len(_plan) - len(_todo)} | to train now: {len(_todo)}")
_t_start, _n_trained = time.time(), 0
for rep in REPS_TO_RUN:
    if rep == "R7" and not R7_OK:
        SKIPPED_REPS[rep] = "continuity hook 7 failed (chunks are not continuous)"
        print(f"\n### {rep} SKIPPED: {SKIPPED_REPS[rep]}")
        continue
    runs = [(k, s) for s in SEEDS_TO_RUN for k in FOLDS_TO_RUN]                # seed-major, as in the reference
    need = [(k, s) for k, s in runs if (rep, k, s) not in _done]
    means_file = MEANS_DIR / f"{rep}.npy"
    if not need and means_file.exists():
        print(f"\n### {rep}: all {len(runs)} runs already logged — skipped")
        continue
    CUR_REP = rep
    _tb = time.time()
    X_ALL, _means = build_rep(rep, NAMES, _starts_of, load_cached)
    np.save(means_file, _means)
    D = REP_DIMS[rep]
    hook(4, X_ALL.shape == (len(WMETA), TA, D) and bool(np.isfinite(X_ALL).all()),
         f"{rep}: X_ALL {X_ALL.shape} (expected ({len(WMETA)}, {TA}, {D})), finite {bool(np.isfinite(X_ALL).all())} | "
         f"built in {time.time() - _tb:.0f} s")
    _ids_hash = hashlib.sha1("\n".join(WMETA.window_id.values[np.concatenate([MISSION_ROWS[n] for n in NAMES])])
                             .encode()).hexdigest()[:16]
    REP_PREP, _zok = prep_rep(X_ALL)
    hook(5, _ids_hash == WINDOW_HASH and len(X_ALL) == len(WMETA)
         and all(REP_PREP[k]["split_hash"] == FOLD_BASE[k]["split_hash"] for k in FOLDS),
         f"{rep}: window ids {_ids_hash} == {WINDOW_HASH}; train/val/test missions per fold == split hash {SPLIT_HASH}",
         stop=True)
    hook(3, _zok, f"{rep}: z-score mean/std from each fold's TRAINING windows only (hashes "
                  f"{[REP_PREP[k]['zhash'] for k in sorted(REP_PREP)]})")
    if rep == "R0" and CHECK_DETERMINISM and DETERMINISM is None:
        run_determinism_check()
    for k, s in runs:
        if (rep, k, s) in _done:
            r = _done[(rep, k, s)]
            print(f"[{rep}] fold {k} ({r.fold_day}) seed {s} — already in CSV (macro-F1 {r.macro_f1:.4f}), skipped")
            continue
        res = train_run(("day_out", k), s, epochs=EPOCHS_TO_RUN)
        append_row(res)
        _n_trained += 1
        _el = time.time() - _t_start
        _left = len(_todo) - _n_trained
        print(f"[{rep} | run {_n_trained}/{len(_todo)}] fold {k} ({res['fold_day']}) seed {s} | window macro-F1 "
              f"{res['macro_f1']:.4f} | mission macro-F1 {res['mission_macro_f1']:.4f} | best epoch {res['best_epoch']} | "
              f"{res['train_seconds']:.0f} s | ETA {_left * _el / _n_trained / 60:.0f} min for {_left} runs")
    del X_ALL, REP_PREP
    gc.collect()
    torch.cuda.empty_cache()
print(f"\nTraining session finished in {(time.time() - _t_start) / 60:.1f} min | runs in CSV: {len(load_done())}/{len(_plan)}"
      + (f" | SKIPPED: {SKIPPED_REPS}" if SKIPPED_REPS else ""))

## 5. Pre-registered decision rule (written BEFORE the results code; not changed after seeing results)
- **Primary metric:** pooled out-of-fold **window** macro-F1 from **seed-averaged** probabilities (all 95 missions,
  each tested exactly once on a model that never saw its day).
- **Secondary metric:** mission-level macro-F1 (mission prediction = argmax of the mean window probabilities).
- **Candidates:** R1, R2, R3, R4, R5, R7 (**K = 6**). R6a and R6b are diagnostics only.
- A candidate **"IMPROVES audio"** over R0 only if **all** of the following hold:
  1. pooled paired delta (candidate − R0) ≥ **+0.02**. The measured seed-to-seed noise floor is about 0.016;
  2. the **99%** mission-cluster bootstrap CI of the paired delta excludes 0. Both reps use the **same** resampled
     missions (2000 resamples within class, seed 2028). 99% because K = 6;
  3. delta > 0 in **at least 4 of 5** folds (seed-averaged, per held-out day);
  4. delta > 0 for **all 3** seeds (each seed's own pooled out-of-fold F1).
- **"WORSE"** is the mirror image: delta ≤ −0.02, the 99% CI entirely below 0, delta < 0 in ≥ 4 of 5 folds, and in
  all 3 seeds.
- Otherwise the candidate is **"NOT CREDITED"**.
- Every candidate is reported, whatever the outcome. **A null result is a valid, expected outcome.** No
  representation is added or changed after seeing the results.

In [ ]:
# ==============================================================================
# 7a. RESULT HELPERS — reference functions (verbatim) + E16 pooling / paired comparison / decision rule
# ==============================================================================
def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def per_class_table(y, pred):
    p, r, f, n = precision_recall_fscore_support(y, pred, labels=list(range(N_CLASSES)), zero_division=0)
    return pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": n}, index=[f"class {c}" for c in range(N_CLASSES)])


def plot_cm(ax, cm, title):
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap=SEQ_CMAP, vmin=0, vmax=1)
    ax.grid(False)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{int(cm[i, j])}\n{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.6 else "#0b0b0b")
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xlabel("predicted class")
    ax.set_ylabel("true class")
    ax.set_title(title, fontsize=9)


def fmt_ms(v):
    v = np.asarray(v, dtype=float)
    return f"{np.nanmean(v):.4f} ± {np.nanstd(v, ddof=1) if len(v) > 1 else 0:.4f}"


def cluster_bootstrap(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Resample MISSIONS within class (never windows). Returns bootstrap confusion matrices (B, 5, 5)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    cnt = np.zeros((len(ms), N_CLASSES), int)
    pos = {m: i for i, m in enumerate(ms)}
    mcls = np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        draw = rows[rng.integers(0, len(rows), (B, len(rows)))]
        cms[:, c, :] = cnt[draw].sum(1)
    return cms


def boot_summary(missions, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    p, r, f = prf_from_cm(cm)
    cms = cluster_bootstrap(missions, y, pred)
    bp, br, bf = zip(*(prf_from_cm(c) for c in cms))
    bp, br, bf = np.array(bp), np.array(br), np.array(bf)
    ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
    tab = pd.DataFrame({"precision": p, "P CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bp).T],
                        "recall": r, "R CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(br).T],
                        "f1": f, "F1 CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bf).T]},
                       index=[f"class {c}" for c in range(N_CLASSES)])
    macro = (f.mean(), *ci(bf.mean(1)))
    bal = (r.mean(), *ci(br.mean(1)))
    return tab, macro, bal, cm, (r, ci(br))


def variance_shares(Zs, y, days):
    """Share of total standardized feature variance explained by class, and by day within class (nested)."""
    n = len(y)
    tot = ((Zs - Zs.mean(0)) ** 2).sum()
    ss_class = sum((y == c).sum() * ((Zs[y == c].mean(0) - Zs.mean(0)) ** 2).sum() for c in np.unique(y))
    ss_day_in_class, df_day = 0.0, 0
    for c in np.unique(y):
        mc = Zs[y == c].mean(0)
        ds = np.unique(days[y == c])
        df_day += len(ds) - 1
        for d in ds:
            s = (y == c) & (days == d)
            ss_day_in_class += s.sum() * ((Zs[s].mean(0) - mc) ** 2).sum()
    ss_day = sum((days == d).sum() * ((Zs[days == d].mean(0) - Zs.mean(0)) ** 2).sum() for d in np.unique(days))
    return {"class": (ss_class / tot, (len(np.unique(y)) - 1) / (n - 1)),
            "day (ignoring class)": (ss_day / tot, (len(np.unique(days)) - 1) / (n - 1)),
            "day within class": (ss_day_in_class / tot, df_day / (n - 1))}


def load_run(rep, k, s):
    z = np.load(PROBS_DIR / f"{rep}_{k}_{s}.npz")
    return {key: z[key] for key in z.files}


def pool_seeds(parts_by_seed):
    """parts_by_seed: {seed: [5 fold dicts with window_id, probs]} -> (sorted window ids, {seed: probs in that order})."""
    out, ids0 = {}, None
    for s, parts in parts_by_seed.items():
        ids = np.concatenate([p["window_id"] for p in parts])
        assert len(ids) == len(set(ids)), f"seed {s}: duplicate windows across folds"
        order = np.argsort(ids)
        out[s] = np.concatenate([p["probs"] for p in parts])[order]
        ids = ids[order]
        assert ids0 is None or np.array_equal(ids, ids0), "seeds do not cover the same windows"
        ids0 = ids
    return ids0, out


def macro_f1(y, pred):
    return float(f1_score(y, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0))


def boot_macro_f1(missions, y, pred):
    """Macro-F1 of every mission-cluster bootstrap resample (reference cluster_bootstrap, seed 2028). With the same
    window order, two representations get IDENTICAL resampled missions, so their difference is paired."""
    return np.array([prf_from_cm(c)[2].mean() for c in cluster_bootstrap(missions, y, pred)])


def rep_stats(per_seed, pool, y):
    """per_seed: {seed: (n_windows, 5)} probabilities aligned with `pool` (DataFrame with mission, day). Primary and
    secondary metrics, per-seed / per-fold F1, safety rates and the bootstrap distribution for paired deltas."""
    P = np.mean(list(per_seed.values()), axis=0)
    pred = P.argmax(1)
    missions = pool.mission.values
    tab, macro, bal, cm, (rec, rec_ci) = boot_summary(missions, y, pred)
    my, mP, mnames = mission_level(missions, y, P)
    _, mmacro, _, _, _ = boot_summary(mnames, my, mP.argmax(1))
    err = pred != y
    return {"P": P, "pred": pred, "window_f1": macro, "window_bal": bal, "cm": cm, "recall": rec,
            "mission_f1": mmacro, "mission_pred": mP.argmax(1), "mission_y": my,
            "seed_f1": {s: macro_f1(y, v.argmax(1)) for s, v in per_seed.items()},
            "fold_f1": {d: macro_f1(y[(pool.day == d).values], pred[(pool.day == d).values]) for d in DAYS},
            "adjacent_share": float((np.abs(pred - y)[err] == 1).mean()) if err.any() else float("nan"),
            "faulty_called_normal": float((pred[y > 0] == 0).mean()),
            "boot": boot_macro_f1(missions, y, pred)}


def decision(delta, lo, hi, folds_pos, folds_neg, seeds_pos, seeds_neg, n_seeds):
    if delta >= MIN_DELTA and lo > 0 and folds_pos >= MIN_FOLDS_SAME_SIGN and seeds_pos == n_seeds:
        return "IMPROVES audio"
    if delta <= -MIN_DELTA and hi < 0 and folds_neg >= MIN_FOLDS_SAME_SIGN and seeds_neg == n_seeds:
        return "WORSE"
    return "NOT CREDITED"


def paired_vs_ref(st, st0):
    """Paired comparison of a representation with R0 under the pre-registered rule."""
    delta = st["window_f1"][0] - st0["window_f1"][0]
    q = (100 - PAIRED_CI) / 2
    lo, hi = np.percentile(st["boot"] - st0["boot"], [q, 100 - q])
    fd = {d: st["fold_f1"][d] - st0["fold_f1"][d] for d in DAYS}
    sd = {s: st["seed_f1"][s] - st0["seed_f1"][s] for s in st0["seed_f1"]}
    fp, fn = sum(v > 0 for v in fd.values()), sum(v < 0 for v in fd.values())
    sp, sn = sum(v > 0 for v in sd.values()), sum(v < 0 for v in sd.values())
    return {"delta": float(delta), "ci_lo": float(lo), "ci_hi": float(hi), "folds_pos": fp, "folds_neg": fn,
            "seeds_pos": sp, "seeds_neg": sn, "fold_delta": fd, "seed_delta": sd,
            "mission_delta": float(st["mission_f1"][0] - st0["mission_f1"][0]),
            "verdict": decision(delta, lo, hi, fp, fn, sp, sn, len(sd))}


def standardize(X):
    sd = X.std(0)
    keep = sd > 1e-12
    return (X[:, keep] - X[:, keep].mean(0)) / sd[keep]


def day_within_class_perm(Zs, y, days, n_perm=VAR_PERMUTATIONS, seed=VAR_PERM_SEED):
    """Chance distribution of the day-within-class share: day labels permuted within each class."""
    rng = np.random.default_rng(seed)
    out = np.empty(n_perm)
    for b in range(n_perm):
        dp = days.copy()
        for c in np.unique(y):
            idx = np.where(y == c)[0]
            dp[idx] = days[rng.permutation(idx)]
        out[b] = variance_shares(Zs, y, dp)["day within class"][0]
    return out

In [ ]:
# ==============================================================================
# 7b. POOLED RESULTS PER REPRESENTATION (seed-averaged, out-of-fold) + PAIRED COMPARISON vs R0 + HOOK 8
# ==============================================================================
_done = load_done()
COMPLETE = [r for r in RUN_ORDER if all((r, k, s) in _done for k in range(len(DAYS)) for s in SEEDS)]
ANALYSIS_READY = (not SMOKE) and "R0" in COMPLETE
STATS, PAIRED = {}, {}
if SMOKE:
    print("SMOKE run: pooled analysis skipped (needs all 5 folds x 3 seeds). Smoke rows:")
    if RESULTS_CSV.exists():
        print(pd.read_csv(RESULTS_CSV)[["rep", "D", "fold", "fold_day", "seed", "macro_f1", "mission_macro_f1",
                                        "best_epoch", "train_seconds"]].to_string(index=False))
elif not ANALYSIS_READY:
    loud("R0 is not complete (5 folds x 3 seeds): the paired analysis cannot run yet. Re-run the training cell.")
else:
    _incomplete = [r for r in REPS_TO_RUN if r not in COMPLETE and r not in SKIPPED_REPS]
    if _incomplete:
        loud(f"Incomplete representations (not analysed): {_incomplete}")
    POOL_IDS = None
    for rep in COMPLETE:
        ids, per_seed = pool_seeds({s: [load_run(rep, k, s) for k in range(len(DAYS))] for s in SEEDS})
        assert len(ids) == len(WMETA), f"{rep}: pooled windows {len(ids)} vs {len(WMETA)}"
        if POOL_IDS is None:
            POOL_IDS = ids
            POOL = WMETA.iloc[[WIN_ID2ROW[w] for w in POOL_IDS]].reset_index(drop=True)
            POOL_Y = POOL.cls.values
        assert np.array_equal(ids, POOL_IDS), f"{rep}: different windows from R0 — not paired"
        STATS[rep] = rep_stats(per_seed, POOL, POOL_Y)
        print(f"{rep}: pooled window macro-F1 {STATS[rep]['window_f1'][0]:.4f} [{STATS[rep]['window_f1'][1]:.4f}, "
              f"{STATS[rep]['window_f1'][2]:.4f}] | mission macro-F1 {STATS[rep]['mission_f1'][0]:.4f}")
    for rep in COMPLETE:
        if rep != "R0":
            PAIRED[rep] = paired_vs_ref(STATS[rep], STATS["R0"])
    _r0 = STATS["R0"]["window_f1"][0]
    hook(8, abs(_r0 - REF_R0_WINDOW_F1[0]) <= REF_TOLERANCE,
         f"R0 pooled window macro-F1 {_r0:.4f} vs the audio-only notebook's {REF_R0_WINDOW_F1[0]:.4f} "
         f"[{REF_R0_WINDOW_F1[1]}, {REF_R0_WINDOW_F1[2]}] (tolerance ±{REF_TOLERANCE})"
         + ("" if abs(_r0 - REF_R0_WINDOW_F1[0]) <= REF_TOLERANCE else
            f" — NOT reproduced: the inner-validation rule probably differed (this run: {INNER_VAL_RULE})"), stop=False)

In [ ]:
# ==============================================================================
# 7c. TABLES: (1) per-representation results, (2) paired vs R0 with the pre-registered verdict
# ==============================================================================
if ANALYSIS_READY:
    _role = lambda r: "control" if r == "R0" else ("candidate" if r in CANDIDATES else "diagnostic")
    _t1 = []
    for rep in COMPLETE:
        st = STATS[rep]
        _t1.append({"rep": rep, "role": _role(rep), "D": REP_DIMS[rep],
                    "window macro-F1 [95% CI]": f"{st['window_f1'][0]:.3f} [{st['window_f1'][1]:.3f}, {st['window_f1'][2]:.3f}]",
                    "mission macro-F1 [95% CI]": f"{st['mission_f1'][0]:.3f} [{st['mission_f1'][1]:.3f}, {st['mission_f1'][2]:.3f}]",
                    **{f"seed {s} F1": round(v, 3) for s, v in st["seed_f1"].items()},
                    "seed SD": round(float(np.std(list(st["seed_f1"].values()), ddof=1)), 4),
                    "min fold F1": round(min(st["fold_f1"].values()), 3),
                    **{f"F1 {d}": round(v, 3) for d, v in st["fold_f1"].items()},
                    **{f"recall c{c}": round(float(st["recall"][c]), 3) for c in range(N_CLASSES)},
                    "adjacent-error share": round(st["adjacent_share"], 3),
                    "faulty called Normal": round(st["faulty_called_normal"], 4)})
    TABLE1 = pd.DataFrame(_t1).set_index("rep")
    TABLE1.to_csv(OUT / "e16_table_results.csv")
    pd.set_option("display.width", 250)
    print("TABLE 1 — pooled out-of-fold results (seed-averaged probabilities; day-out; 95 missions)")
    print(TABLE1.T.to_string())

    _t2 = []
    for rep in [r for r in RUN_ORDER if r in PAIRED]:
        p = PAIRED[rep]
        _t2.append({"rep": rep, "role": _role(rep), "D": REP_DIMS[rep], "delta window F1 (pooled)": round(p["delta"], 4),
                    f"{PAIRED_CI:.0f}% CI": f"[{p['ci_lo']:+.4f}, {p['ci_hi']:+.4f}]",
                    "folds positive (of 5)": p["folds_pos"], "seeds positive (of 3)": p["seeds_pos"],
                    "delta mission F1": round(p["mission_delta"], 4),
                    "verdict": p["verdict"] if rep in CANDIDATES else f"(diagnostic: {p['verdict']})"})
    for rep, why in SKIPPED_REPS.items():
        _t2.append({"rep": rep, "role": _role(rep), "D": REP_DIMS[rep], "verdict": f"SKIPPED — {why}"})
    for rep in [r for r in CANDIDATES if r not in PAIRED and r not in SKIPPED_REPS]:
        _t2.append({"rep": rep, "role": _role(rep), "D": REP_DIMS[rep], "verdict": "NOT RUN / INCOMPLETE"})
    TABLE2 = pd.DataFrame(_t2).set_index("rep")
    TABLE2.to_csv(OUT / "e16_table_paired.csv")
    print(f"\nTABLE 2 — paired vs R0 (same missions resampled for both; {PAIRED_CI:.0f}% CI because K = {len(CANDIDATES)})")
    print(TABLE2.to_string())
    print("\nPer-fold deltas (seed-averaged) and per-seed deltas vs R0:")
    for rep, p in PAIRED.items():
        print(f"  {rep:<4} folds: " + ", ".join(f"{d} {v:+.3f}" for d, v in p["fold_delta"].items())
              + " | seeds: " + ", ".join(f"{s} {v:+.3f}" for s, v in p["seed_delta"].items()))

In [ ]:
# ==============================================================================
# 7d. DESCRIPTIVE ONLY (never used for selection): variance of standardized mission-mean features explained by
#     class and by day-within-class (chance: day labels permuted within class, 200 times)
# ==============================================================================
VAR_TAB = None
if ANALYSIS_READY:
    _vr = []
    for rep in COMPLETE:
        Zs = standardize(np.load(MEANS_DIR / f"{rep}.npy"))
        vs = variance_shares(Zs, Y_M, DAY_M)
        perm = day_within_class_perm(Zs, Y_M, DAY_M)
        _vr.append({"rep": rep, "D (non-constant)": Zs.shape[1], "class share": vs["class"][0],
                    "class chance (df/(n-1))": vs["class"][1], "day-within-class share": vs["day within class"][0],
                    "day-within-class chance (perm mean)": perm.mean(),
                    "day-within-class perm 95th pct": np.percentile(perm, 95),
                    "class / day-within-class": vs["class"][0] / max(vs["day within class"][0], 1e-12)})
    VAR_TAB = pd.DataFrame(_vr).set_index("rep")
    VAR_TAB.to_csv(OUT / "e16_variance_shares.csv")
    print("DESCRIPTIVE ONLY — share of standardized mission-mean feature variance (95 missions; all missions, test "
          "included; NOT used to choose anything):")
    print(VAR_TAB.round(3).to_string())

In [ ]:
# ==============================================================================
# 7e. FIGURES: (1) pooled window macro-F1 with 95% CIs per representation; (2) per-day F1 heat table
# ==============================================================================
if ANALYSIS_READY:
    _reps = [r for r in RUN_ORDER if r in STATS]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    _y = [STATS[r]["window_f1"][0] for r in _reps]
    _err = np.array([[STATS[r]["window_f1"][0] - STATS[r]["window_f1"][1], STATS[r]["window_f1"][2] - STATS[r]["window_f1"][0]]
                     for r in _reps]).T
    _col = [CLASS_COLORS[0] if r == "R0" else (CLASS_COLORS[1] if r in CANDIDATES else CLASS_COLORS[2]) for r in _reps]
    for i, r in enumerate(_reps):
        ax.errorbar(i, _y[i], yerr=_err[:, i:i + 1], fmt="o", ms=8, color=_col[i], capsize=4)
    ax.axhline(STATS["R0"]["window_f1"][0], color="#52514e", lw=1, ls="--")
    ax.axhspan(STATS["R0"]["window_f1"][0] - MIN_DELTA, STATS["R0"]["window_f1"][0] + MIN_DELTA, color="#52514e", alpha=0.08)
    ax.set_xticks(range(len(_reps)))
    ax.set_xticklabels([f"{r}\nD={REP_DIMS[r]}" for r in _reps])
    ax.set_ylabel("pooled window macro-F1 (day-out)")
    ax.set_title("E16: pooled out-of-fold window macro-F1 per representation (blue = R0 control, orange = candidate, "
                 "green = diagnostic)")
    save_fig(fig, "e16_fig1_pooled_f1",
             "Seed-averaged probabilities, 95 missions, each tested once on an unseen day. Bars = 95% mission-cluster "
             "bootstrap CIs (unpaired); dashed line = R0, shaded band = R0 ± 0.02 (the rule's minimum delta). The paired "
             "99% CIs that decide the verdict are in Table 2.")

    _heat = np.array([[STATS[r]["fold_f1"][d] for d in DAYS] for r in _reps])
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(_reps) + 1.6))
    ax.imshow(_heat, cmap="Blues", vmin=max(0.0, _heat.min() - 0.05), vmax=1.0, aspect="auto")
    ax.grid(False)
    for i in range(len(_reps)):
        for j in range(len(DAYS)):
            ax.text(j, i, f"{_heat[i, j]:.3f}", ha="center", va="center", fontsize=8,
                    color="white" if _heat[i, j] > 0.85 else "#0b0b0b")
    ax.set_xticks(range(len(DAYS)))
    ax.set_xticklabels([d + (" (10 missions)" if d == SMALL_FOLD_DAY else "") for d in DAYS], fontsize=8)
    ax.set_yticks(range(len(_reps)))
    ax.set_yticklabels(_reps)
    ax.set_title("Window macro-F1 per held-out day (seed-averaged)")
    save_fig(fig, "e16_fig2_per_day_f1",
             "Each cell = window macro-F1 on one unseen recording day. Each day has only 2-8 missions per class; "
             "2021-12-22 has 10 missions in total. No frequency axis is shown; all features are per-channel FFT bins "
             "(unverified Hz).")

In [ ]:
# ==============================================================================
# 8. WHAT E16 TELLS US (generated from the numbers) -> e16_summary.txt; e16_outputs.zip; final hook table
# ==============================================================================
import zipfile

_L = ["WHAT E16 TELLS US", ""]
if SMOKE:
    _L += ["SMOKE run only (R0 + R5, fold 1, seed 42, 1 epoch): checks that the pipeline runs end to end. "
           "No scientific conclusion can be drawn from it."]
elif not ANALYSIS_READY:
    _L += ["The analysis did not run: R0 is incomplete. Re-run the training cell (it resumes)."]
else:
    _cands = [r for r in CANDIDATES if r in PAIRED]
    _pass = [r for r in _cands if PAIRED[r]["verdict"] == "IMPROVES audio"]
    _worse = [r for r in _cands if PAIRED[r]["verdict"] == "WORSE"]
    _L += [f"Representations analysed: {COMPLETE}; candidates compared with R0: {_cands} "
           f"(K = {len(CANDIDATES)} pre-registered)" + (f"; skipped: {SKIPPED_REPS}" if SKIPPED_REPS else "") + ".",
           f"R0 (control) pooled window macro-F1 {STATS['R0']['window_f1'][0]:.3f} [{STATS['R0']['window_f1'][1]:.3f}, "
           f"{STATS['R0']['window_f1'][2]:.3f}], mission macro-F1 {STATS['R0']['mission_f1'][0]:.3f}.",
           ""]
    _L += [f"- {r}: delta {PAIRED[r]['delta']:+.3f} [{PAIRED[r]['ci_lo']:+.3f}, {PAIRED[r]['ci_hi']:+.3f}] ({PAIRED_CI:.0f}% "
           f"paired CI), folds positive {PAIRED[r]['folds_pos']}/5, seeds positive {PAIRED[r]['seeds_pos']}/{len(SEEDS)}, "
           f"mission delta {PAIRED[r]['mission_delta']:+.3f} -> {PAIRED[r]['verdict']}" for r in _cands]
    _L += [f"- {r} (diagnostic): delta {PAIRED[r]['delta']:+.3f} [{PAIRED[r]['ci_lo']:+.3f}, {PAIRED[r]['ci_hi']:+.3f}]"
           for r in DIAGNOSTIC_REPS if r in PAIRED]
    _L += ["",
           ("Candidates that pass the pre-registered rule: " + ", ".join(_pass) + "." if _pass else
            "No candidate passes the pre-registered rule."),
           ("Candidates that are WORSE than R0 under the mirror rule: " + ", ".join(_worse) + "." if _worse else
            "No candidate is credited as WORSE."),
           ("ANSWER (from the rule only): the input representation DOES limit audio-only day-out classification; "
            f"{', '.join(_pass)} improve(s) on R0." if _pass else
            "ANSWER (from the rule only): this experiment gives NO evidence that the input representation limits audio-only "
            "day-out classification: no candidate improves on R0 by the pre-registered criteria."),
           "",
           "Worst unseen day per representation (seed-averaged window macro-F1):"]
    _L += [f"- {r}: {min(STATS[r]['fold_f1'], key=STATS[r]['fold_f1'].get)} "
           f"({min(STATS[r]['fold_f1'].values()):.3f}); best {max(STATS[r]['fold_f1'], key=STATS[r]['fold_f1'].get)} "
           f"({max(STATS[r]['fold_f1'].values()):.3f})" for r in COMPLETE]
    if VAR_TAB is not None:
        _L += ["", "Descriptive (not used for selection): day-within-class share of mission-mean feature variance vs "
                   "permutation chance: " + ", ".join(f"{r} {VAR_TAB.loc[r, 'day-within-class share']:.2f} "
                                                      f"(chance {VAR_TAB.loc[r, 'day-within-class chance (perm mean)']:.2f})"
                                                      for r in VAR_TAB.index) + "."]
    _h8 = [ok for ok, _ in HOOKS.get(8, [])]
    _L += ["", f"R0 reproduction of the audio-only notebook (hook 8, soft): {'PASS' if all(_h8) and _h8 else 'FAIL'} "
               f"(inner-validation rule: {INNER_VAL_RULE})."]
_L += ["",
       "Limits",
       "- 95 missions (~19 per class); per-class recall is known only to about ±0.1.",
       f"- {len(SEEDS)} seeds; the 2021-12-22 fold has only 10 missions.",
       "- No rotor RPM: order-normalised / harmonic representations were out of scope.",
       "- The per-channel sample rate is unverified; every frequency is a per-channel FFT bin, never Hz.",
       f"- K = {len(CANDIDATES)} candidate comparisons (99% CIs); one model, optimizer, window length and split were held "
       "fixed, so the result is about the input representation under this pipeline only.",
       "- Day-out pooled out-of-fold metrics; no representation was tuned."]
SUMMARY = "\n".join(_L)
print(SUMMARY)
(OUT / ("e16_summary_smoke.txt" if SMOKE else "e16_summary.txt")).write_text(SUMMARY, encoding="utf-8")

# ---- e16_outputs.zip: CSVs, summary, figures, mission means, probabilities (never the cache, checkpoints or dataset) ----
_zip = OUT / ("e16_outputs_smoke.zip" if SMOKE else "e16_outputs.zip")
_n_files = 0
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("e16_*.csv")) + sorted(OUT.glob("e16_summary*.txt")) + sorted(FIG_DIR.glob("*.png")) \
            + sorted(MEANS_DIR.glob("*.npy")) + (sorted(PROBS_DIR.glob("*.npz")) if ZIP_PROBS else []):
        zf.write(p, p.relative_to(OUT))
        _n_files += 1
print(f"\nWrote {_zip} ({_n_files} files, {_zip.stat().st_size / 1e6:.1f} MB; cache/checkpoints/dataset excluded)")

# ---- file-access summary (reference), then every hook ----
print("\nFile-access summary (every dataset file read goes through open_logged):")
for k in ACCESS_KINDS:
    print(f"  {k:<40} {FILE_LOG[k]:>4} opens, {len(FILE_LOG_PER_MISSION[k]):>3} distinct missions")
if DETERMINISM is None:
    print("  Determinism check: NOT RUN (SMOKE, CHECK_DETERMINISM=False, or R0 already complete in an earlier session)")
print("\n" + "=" * 110)
print("VERIFICATION HOOKS")
print("=" * 110)
for n in range(1, 9):
    res = HOOKS.get(n, [])
    status = "NOT RUN" if not res else ("PASS" if all(ok for ok, _ in res) else "FAIL")
    print(f"  HOOK {n} {status:<8} {HOOK_NAMES[n]}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else 'FAIL'}: {msg}")
print("=" * 110)
print("\n💾 Click \"Save Version\" -> \"Save & Run All (Commit)\" to keep /kaggle/working (or download e16_outputs.zip).")